In [3]:
import pandas as pd

df = pd.read_csv('survey_results_public.csv', low_memory=False)
print(df.shape)


# %%
keep = ['AISelect', 'AIAcc', 'AISent', 'AIThreat', 'WorkExp',
        'DevType', 'OrgSize', 'Country', 'ConvertedCompYearly']
df = df[keep]
print(df.shape)

# %%
df = df.dropna(subset=['AIAcc', 'AISelect'])
print(df.shape)

# %%

# Block 4 — shorten the long answer labels
usage_map = {
    'Yes, I use AI tools daily': '1 Daily',
    'Yes, I use AI tools weekly': '2 Weekly',
    'Yes, I use AI tools monthly or infrequently': '3 Infrequent',
    'No, but I plan to soon': '4 Not yet',
    "No, and I don't plan to": '5 Never',
}
df['ai_usage'] = df['AISelect'].map(usage_map)

trust_map = {
    'Highly trust': '1 Highly trust',
    'Somewhat trust': '2 Somewhat trust',
    'Neither trust nor distrust': '3 Neutral',
    'Somewhat distrust': '4 Somewhat distrust',
    'Highly distrust': '5 Highly distrust',
}
df['ai_trust'] = df['AIAcc'].map(trust_map)

print(df['ai_usage'].value_counts())

# Block 5 — experience bands

# %%
df['WorkExp'] = pd.to_numeric(df['WorkExp'], errors='coerce')

def experience_band(years):
    if pd.isna(years):
        return None
    if years < 3:
        return '1 Early (0-2 yrs)'
    if years < 10:
        return '2 Mid (3-9 yrs)'
    if years < 20:
        return '3 Senior (10-19 yrs)'
    return '4 Veteran (20+ yrs)'

df['experience_band'] = df['WorkExp'].apply(experience_band)
print(df['experience_band'].value_counts())

#Block 6 — trust flags

df['trusts_ai'] = df['ai_trust'].isin(['1 Highly trust', '2 Somewhat trust']).astype(int)
df['distrusts_ai'] = df['ai_trust'].isin(['4 Somewhat distrust', '5 Highly distrust']).astype(int)
df['uses_ai_daily'] = (df['ai_usage'] == '1 Daily').astype(int)
print(df['trusts_ai'].mean())

#Block 7 — clean the salary

df['salary_usd'] = pd.to_numeric(df['ConvertedCompYearly'], errors='coerce')
df.loc[(df['salary_usd'] < 5000) | (df['salary_usd'] > 400000), 'salary_usd'] = None
print(df['salary_usd'].notna().sum())

# Block 8 — group job roles

top_roles = df['DevType'].value_counts().head(10).index
df['dev_role'] = df['DevType'].where(df['DevType'].isin(top_roles), 'Other')
print(df['dev_role'].value_counts())

#Block 9 — export

final = df[['ai_usage', 'ai_trust', 'trusts_ai', 'distrusts_ai', 'uses_ai_daily',
            'experience_band', 'dev_role', 'salary_usd', 'Country', 'OrgSize',
            'AISent', 'AIThreat']].rename(columns={
    'Country': 'country', 'OrgSize': 'org_size',
    'AISent': 'ai_sentiment', 'AIThreat': 'ai_job_threat'})

final.to_csv('ai_trust_clean.csv', index=False)
print(final.shape)


(49191, 172)
(49191, 9)
(33260, 9)
ai_usage
1 Daily         15662
2 Weekly         5901
5 Never          5380
3 Infrequent     4563
4 Not yet        1754
Name: count, dtype: int64
experience_band
2 Mid (3-9 yrs)         9657
3 Senior (10-19 yrs)    9417
4 Veteran (20+ yrs)     8646
1 Early (0-2 yrs)       3026
Name: count, dtype: int64
0.32784125075165366
21272
dev_role
Developer, full-stack                            9438
Other                                            7353
Developer, back-end                              4873
Student                                          2229
Architect, software or solutions                 2120
Developer, desktop or enterprise applications    1545
Developer, front-end                             1463
Other (please specify):                          1351
Developer, mobile                                1020
Developer, embedded applications or devices      1011
Academic researcher                               857
Name: count, dtype: int64
(33260,